# Smart AI Advisory System — Data Preparation & Processing (Disease Detection)

This notebook documents the **data preprocessing stage for disease detection only**.

It is aligned with the current dataset, which contains **leaf images for supported crops/classes used by the disease model**.

**Topics covered:**
1. Dataset structure (class folders)
2. Label mapping
3. Train/validation/test split
4. Image preprocessing transforms for training and inference
5. Treatment metadata mapping

---
## 1. Disease Detection Data Pipeline

Our disease detection model uses a **PlantVillage-style** dataset: one folder per disease class, each containing leaf images (e.g. `.JPG`, `.jpg`, `.png`).

### 1.1 Dataset structure

Expected layout:
```
dataset/plant_village/
├── Tomato___Early_blight/
│   ├── img1.JPG
│   └── ...
├── Tomato___Late_blight/
├── Tomato___Healthy/
├── Potato___Early_blight/
└── ...
```

Each folder name is the **class label**; we discover classes by listing subdirectories.

In [1]:
import json
from pathlib import Path

# Paths (relative to project root)
PROJECT_ROOT = Path('..')
MODELS_DIR = PROJECT_ROOT / 'models'
DATASET_DIR = PROJECT_ROOT / 'dataset' / 'plant_village'

label_map = None
label_map_path = MODELS_DIR / 'label_map.json'

if label_map_path.exists():
    with open(label_map_path, 'r', encoding='utf-8') as f:
        label_map = json.load(f)
    print('Label map loaded from models/label_map.json')
else:
    print('label_map.json not found. Building a temporary class map from dataset folders...')
    if DATASET_DIR.exists():
        class_folders = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir()])
        label_map = {str(i): name for i, name in enumerate(class_folders)}
    else:
        label_map = {}

print('\nLabel map (class index -> class name):')
for idx, name in sorted(label_map.items(), key=lambda x: int(x[0])):
    print(f'  {idx}: {name}')
print(f'\nTotal classes: {len(label_map)}')

Label map loaded from models/label_map.json

Label map (class index -> class name):
  0: Pepper__bell___Bacterial_spot
  1: Pepper__bell___healthy
  2: Potato___Early_blight
  3: Potato___healthy
  4: Potato___Late_blight
  5: Tomato__Target_Spot
  6: Tomato__Tomato_mosaic_virus
  7: Tomato__Tomato_YellowLeaf__Curl_Virus
  8: Tomato_Bacterial_spot
  9: Tomato_Early_blight
  10: Tomato_healthy
  11: Tomato_Late_blight
  12: Tomato_Leaf_Mold
  13: Tomato_Septoria_leaf_spot
  14: Tomato_Spider_mites_Two_spotted_spider_mite

Total classes: 15


### 1.2 Discovering classes and splitting the dataset

We:
1. **Discover classes** by scanning subdirectories (each folder = one disease/healthy class).  
2. **Split images** per class into **train (70%)**, **validation (15%)**, and **test (15%)** using a fixed random seed for reproducibility.  
3. **Build lists** of `(image_path, class_index)` for each split.

Below we replicate the logic from `backend/disease_detection/preprocess.py`.

In [2]:
from pathlib import Path
import random

try:
    from sklearn.model_selection import train_test_split
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


def get_class_folders(dataset_dir):
    """Get all class folders from dataset directory."""
    dataset_path = Path(dataset_dir)
    if not dataset_path.exists():
        raise ValueError(f"Dataset directory not found: {dataset_dir}")
    class_folders = [d for d in dataset_path.iterdir() if d.is_dir()]
    return sorted(class_folders)


def simple_split(files, train_ratio=0.7, val_ratio=0.15, seed=42):
    """Fallback split when sklearn is unavailable."""
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    n = len(files)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = files[:n_train]
    val = files[n_train:n_train + n_val]
    test = files[n_train + n_val:]
    return train, val, test


def split_dataset(dataset_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """Split dataset into train, validation, and test sets."""
    class_folders = get_class_folders(dataset_dir)
    train_data, val_data, test_data = [], [], []

    for class_idx, folder in enumerate(class_folders):
        image_files = (
            list(folder.glob('*.JPG')) +
            list(folder.glob('*.jpg')) +
            list(folder.glob('*.png')) +
            list(folder.glob('*.jpeg')) +
            list(folder.glob('*.JPEG')) +
            list(folder.glob('*.webp'))
        )

        if len(image_files) < 3:
            # Keep tiny classes in train split to avoid split errors
            train = image_files
            val = []
            test = []
        elif SKLEARN_AVAILABLE:
            train, temp = train_test_split(image_files, test_size=(1 - train_ratio), random_state=42)
            val, test = train_test_split(temp, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=42)
        else:
            train, val, test = simple_split(image_files, train_ratio=train_ratio, val_ratio=val_ratio, seed=42)

        train_data.extend([(str(img), class_idx) for img in train])
        val_data.extend([(str(img), class_idx) for img in val])
        test_data.extend([(str(img), class_idx) for img in test])

    return train_data, val_data, test_data


dataset_dir = PROJECT_ROOT / 'dataset' / 'plant_village'
if dataset_dir.exists():
    train_data, val_data, test_data = split_dataset(dataset_dir)
    print(f'Split complete (sklearn available: {SKLEARN_AVAILABLE})')
    print(f'Train samples: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)}')
else:
    print('Dataset directory not found at ../dataset/plant_village')
    print('Notebook still runs, but please place your image dataset there to execute the full preprocessing flow.')

Split complete (sklearn available: True)
Train samples: 28883, Validation: 6191, Test: 6201


### 1.3 Image preprocessing for training and inference

All images are:
1. **Resized** to **224×224** (input size for MobileNetV2).  
2. **Normalized** with ImageNet statistics:  
   - mean = [0.485, 0.456, 0.406]  
   - std = [0.229, 0.224, 0.225]  

**Training only:** we add data augmentation (random horizontal flip, rotation, color jitter) to improve generalization.

In [3]:
# Dependency check only (no installation inside notebook)
try:
    import torch
    import torchvision
    print(f'torch: {torch.__version__}')
    print(f'torchvision: {torchvision.__version__}')
except Exception:
    print('torch/torchvision not available in this kernel.')
    print('Install project dependencies from requirements.txt in your virtual environment, then rerun.')


torch: 2.9.0+cpu
torchvision: 0.24.0+cpu


In [4]:
# Same normalization as in train.py and infer.py
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

try:
    import torchvision.transforms as T

    # Validation/Test (no augmentation)
    val_test_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    # Training (with augmentation)
    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    print('Preprocessing pipeline:')
    print('  - Resize: 224x224')
    print('  - Normalize: ImageNet mean/std')
    print('  - Training: + RandomHorizontalFlip, RandomRotation(15 deg), ColorJitter')
except Exception:
    print('torchvision is not available in this kernel.')
    print('Transforms shown conceptually: Resize(224,224) + Normalize(mean,std) + augmentation for training.')

Preprocessing pipeline:
  - Resize: 224x224
  - Normalize: ImageNet mean/std
  - Training: + RandomHorizontalFlip, RandomRotation(15 deg), ColorJitter


### 1.4 Disease treatments (metadata)

We map each predicted class to **treatment text** (general, prevention, organic) so the app can show actionable advice. This is stored in `disease_treatments.json`.

In [5]:
treatments_path = MODELS_DIR / 'disease_treatments.json'
if treatments_path.exists():
    with open(treatments_path, 'r', encoding='utf-8') as f:
        treatments = json.load(f)

    example_key = list(treatments.keys())[0]
    print(f'Example entry for "{example_key}":')
    for key, value in list(treatments[example_key].items())[:2]:
        print(f'  {key}: {value[:80]}...' if isinstance(value, str) and len(value) > 80 else f'  {key}: {value}')
    print(f'\nTotal treatment entries: {len(treatments)}')
else:
    print('disease_treatments.json not found in ../models')
    print('Train/export pipeline should generate this metadata before full app inference.')

Example entry for "Tomato___Early_blight":
  general: Early blight is caused by Alternaria solani. Remove infected leaves immediately....
  prevention: Use disease-resistant varieties. Rotate crops annually. Water at the base of pla...

Total treatment entries: 32


---
## 2. Scope Note

This notebook is intentionally limited to **image data preprocessing for disease detection**.

Market-price and RAG processing are implemented in backend modules, but omitted here so this notebook strictly represents the dataset used to train the crop-disease model.

### 2.1 Out-of-scope modules

Market and RAG pipelines are documented and implemented in backend services.

They are intentionally excluded from this notebook to keep this notebook focused on **image preprocessing for disease detection training data only**.

In [6]:
# Intentionally no executable market-preprocessing code in this notebook.
# This notebook is scoped to disease-image preprocessing only.
print('Skipped: market preprocessing examples are out of scope for this notebook.')

Skipped: market preprocessing examples are out of scope for this notebook.


### 2.2 Note

Prediction examples for market time-series are intentionally omitted from this notebook to keep preprocessing scope aligned to disease-image training data only.

In [7]:
# Intentionally no executable market-prediction code in this notebook.
print('Skipped: market prediction preprocessing is out of scope for this disease-only data notebook.')

Skipped: market prediction preprocessing is out of scope for this disease-only data notebook.


---
## 3. Scope Note

RAG and market analytics are outside this notebook’s scope.

This notebook is intentionally focused on **preprocessing image data used for disease detection model training**.

### 3.1 Out-of-scope modules

For RAG and market modules, refer to backend code and project documentation. No additional preprocessing is executed here.

In [8]:
# Intentionally no executable RAG preprocessing code in this notebook.
print('Skipped: RAG preprocessing is out of scope for this disease-image data notebook.')

Skipped: RAG preprocessing is out of scope for this disease-image data notebook.


### 3.2 Note

Embedding and retrieval steps are handled in backend services and are not part of this notebook’s image-preprocessing execution path.

In [9]:
# Intentionally no optional model downloads/imports here.
# Keeping this notebook deterministic and focused on disease-image preprocessing only.
print('Skipped: sentence-transformers demo removed from this notebook scope.')

Skipped: sentence-transformers demo removed from this notebook scope.


---
## Summary

| Component | Data source / structure | Main preprocessing steps |
|---|---|---|
| **Disease detection (only)** | Folder-per-class leaf images (`dataset/plant_village`) | Class discovery → label map → 70/15/15 split → resize 224x224 → ImageNet normalization → optional augmentation → treatment metadata mapping |

This notebook is intentionally scoped to the image preprocessing stage used by:
- `backend/disease_detection/preprocess.py`
- `backend/disease_detection/train.py`
- `backend/disease_detection/infer.py`

For market and RAG modules, use backend service code and docs; they are not part of this dataset preprocessing notebook.